In [20]:
import os
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import random
from google.colab import drive
from torchvision import datasets, transforms


In [21]:
# Monter Google Drive et charger la dataset chest_xray
drive.mount('/content/drive')

data_dir = "/content/drive/MyDrive/chest_xray"

batch_size_train = 32
batch_size_test = 32
image_size = 128

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.ImageFolder(
    root=os.path.join(data_dir, "train"),
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    root=os.path.join(data_dir, "test"),
    transform=test_transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size_train, shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size_test, shuffle=True
)

class_names = train_dataset.classes
print("Classes :", class_names)
print("Nombre d'images train :", len(train_dataset))
print("Nombre d'images test :", len(test_dataset))


In [ ]:
# Let's draw some of the training data
examples = enumerate(test_loader)
batch_idx, (example_data, example_targets) = next(examples)

fig = plt.figure()
for i in range(6):
  plt.subplot(2,3,i+1)
  plt.tight_layout()
  plt.imshow(example_data[i][0], cmap='gray', interpolation='none')
  plt.title("Ground Truth: {}".format(example_targets[i]))
  plt.xticks([])
  plt.yticks([])
plt.show()

Define the network.  This is a more typical way to define a network than the sequential structure.  We define a class for the network, and define the parameters in the constructor.  Then we use a function called forward to actually run the network.  It's easy to see how you might use residual connections in this format.

In [ ]:
# CNN adapté à chest_xray
# Architecture demandée :
# 1. Conv valide k=5, 1 canal en entrée, 10 canaux en sortie
# 2. MaxPool 2x2
# 3. ReLU
# 4. Conv valide k=5, 10 canaux en entrée, 20 canaux en sortie
# 5. Dropout2d
# 6. MaxPool 2x2
# 7. ReLU
# 8. Flatten
# 9. Fully connected vers 50
# 10. ReLU
# 11. Fully connected vers 2 (NORMAL / PNEUMONIA)
# 12. LogSoftmax

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=10, kernel_size=5)
        self.conv2 = nn.Conv2d(in_channels=10, out_channels=20, kernel_size=5)
        self.dropout = nn.Dropout2d()
        self.fc1 = nn.Linear(20 * 29 * 29, 50)
        self.fc2 = nn.Linear(50, 2)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.dropout(self.conv2(x)), 2))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)


In [24]:
# He initialization of weights
def weights_init(layer_in):
  if isinstance(layer_in, (nn.Linear, nn.Conv2d)):
    nn.init.kaiming_uniform_(layer_in.weight)
    if layer_in.bias is not None:
      layer_in.bias.data.fill_(0.0)


In [25]:
# Create network
model = Net()
# Initialize model weights
model.apply(weights_init)
# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [26]:
# Main training routine
def train(epoch):
  model.train()
  # Get each
  for batch_idx, (data, target) in enumerate(train_loader):
    optimizer.zero_grad()
    output = model(data)
    loss = F.nll_loss(output, target)
    loss.backward()
    optimizer.step()
    # Store results
    if batch_idx % 10 == 0:
      print('Train Epoch: {} [{}/{}]\tLoss: {:.6f}'.format(
        epoch, batch_idx * len(data), len(train_loader.dataset), loss.item()))

In [27]:
# Run on test data
def test():
  model.eval()
  test_loss = 0
  correct = 0
  with torch.no_grad():
    for data, target in test_loader:
      output = model(data)
      test_loss += F.nll_loss(output, target, reduction='sum').item()
      pred = output.data.max(1, keepdim=True)[1]
      correct += pred.eq(target.data.view_as(pred)).sum().item()
  test_loss /= len(test_loader.dataset)
  print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
    test_loss, correct, len(test_loader.dataset),
    100. * correct / len(test_loader.dataset)))


In [ ]:
# Get initial performance
test()
# Train for a few epochs
n_epochs = 5
for epoch in range(1, n_epochs + 1):
  train(epoch)
  test()


In [ ]:
# Run network on data we got before and show predictions
output = model(example_data)
preds = output.data.max(1, keepdim=True)[1]

fig = plt.figure(figsize=(10, 10))
for i in range(min(10, len(example_data))):
  plt.subplot(5, 5, i + 1)
  plt.tight_layout()
  plt.imshow(example_data[i][0], cmap='gray', interpolation='none')
  plt.title("Pred: {}".format(class_names[preds[i].item()]))
  plt.xticks([])
  plt.yticks([])
plt.show()
